# 05 — Prompt Variance

Systematic evaluation of how prompt design affects GPT-5 credit risk predictions.

**Fixed:** model (GPT-5), dataset (100-loan sample, no borrower description), evaluation metrics  
**Varied:** system prompt framing and user prompt structure

**Variants tested:**

| # | Name | What changes |
|---|------|--------------|
| 0 | `baseline` | Current production prompt — control group |
| 1 | `conservative` | Role reframed as risk-averse underwriter; bias toward flagging defaults |
| 2 | `chain_of_thought` | Instructed to reason step-by-step through risk factors before predicting |
| 3 | `few_shot` | 4 labeled training examples prepended to each user prompt |
| 4 | `top_features_only` | Only the 8 most important features (by XGBoost importance) passed in |
| 5 | `structured_4factor` | Explicit 4-factor evaluation framework in system prompt |

**Outputs:**
- `data/results/llm/05_prompt_variance_metrics.csv` — quantitative comparison (accuracy, AUC, F1)
- `data/results/llm/05_prompt_variance_predictions.csv` — per-loan predictions + reasonings per variant
- `data/results/llm/05_prompt_variance_reasonings.jsonl` — reasoning text per variant, for promptfoo qualitative eval

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from llm_utils import (
    load_llm_sample,
    run_ml_on_sample,
    run_llm_experiment,
    evaluate_predictions,
    build_system_prompt,
    build_user_prompt,
    build_few_shot_examples,
    format_loan_features,
    FEATURE_DESCRIPTIONS,
    RESULTS_DIR,
)

## Define Prompt Variants

In [ ]:
# ── Top 8 features by XGBoost importance (used in variant 4) ────────────────
TOP_FEATURES = [
    'int_rate', 'sub_grade', 'dti', 'revol_util',
    'annual_inc', 'installment', 'loan_amnt', 'revol_bal',
]

def format_top_features(row):
    lines = []
    for feat in TOP_FEATURES:
        if feat in row and pd.notna(row[feat]):
            label = FEATURE_DESCRIPTIONS.get(feat, feat)
            lines.append(f"- {label}: {row[feat]}")
    return "\n".join(lines)


# ── Shared JSON output instruction (appended to all system prompts) ──────────
_JSON_INSTRUCTION = (
    "\n\nRespond ONLY with valid JSON in this exact format:\n"
    '{"prediction": <1 or 0>, "reasoning": "<brief explanation>"}\n\n'
    "Where:\n"
    "- prediction: 1 = Fully Paid, 0 = Charged Off\n"
    "- reasoning: 1-2 sentence explanation of your prediction"
)


# ── Variant definitions ──────────────────────────────────────────────────────
# Each entry: name, system_prompt (str), user_prompt_fn (callable(row) -> str)

PROMPT_VARIANTS = [
    {
        "name": "baseline",
        "description": "Current production prompt — control group",
        "system_prompt": build_system_prompt(),
        "user_prompt_fn": lambda row: build_user_prompt(row, include_desc=False),
    },
    {
        "name": "conservative",
        "description": "Risk-averse underwriter; bias toward flagging defaults when uncertain",
        "system_prompt": (
            "You are a risk-averse credit underwriter at a bank. "
            "Your primary responsibility is protecting the institution from loan defaults. "
            "When the evidence is mixed or uncertain, err on the side of caution and predict Charged Off. "
            "Given a loan application's features, predict whether the borrower will fully repay or default."
            + _JSON_INSTRUCTION
        ),
        "user_prompt_fn": lambda row: build_user_prompt(row, include_desc=False),
    },
    {
        "name": "chain_of_thought",
        "description": "Explicit step-by-step reasoning through risk factors before prediction",
        "system_prompt": (
            "You are a credit risk analyst. "
            "Before predicting loan outcomes, reason step by step through the evidence. "
            "Identify the key risk signals, weigh them against protective factors, "
            "then arrive at a final prediction."
            + _JSON_INSTRUCTION.replace(
                "1-2 sentence explanation of your prediction",
                "step-by-step analysis: risk signals, protective factors, and conclusion"
            )
        ),
        "user_prompt_fn": lambda row: (
            "Analyze this loan step by step and predict its outcome:\n\n"
            + format_loan_features(row, include_desc=False)
        ),
    },
    {
        "name": "few_shot",
        "description": "4 labeled training examples prepended before each prediction request",
        "system_prompt": build_system_prompt(),
        "user_prompt_fn": None,  # set after building few-shot text below
    },
    {
        "name": "top_features_only",
        "description": "Only the 8 most important features by XGBoost importance are passed in",
        "system_prompt": build_system_prompt(),
        "user_prompt_fn": lambda row: (
            "Predict the outcome for this loan application "
            "(key features only):\n\n"
            + format_top_features(row)
        ),
    },
    {
        "name": "structured_4factor",
        "description": "Explicit 4-factor evaluation framework: income capacity, debt burden, credit history, loan characteristics",
        "system_prompt": (
            "You are a credit risk analyst. "
            "Evaluate loans using this 4-factor framework:\n"
            "1. Income & Repayment Capacity: annual income vs. monthly installment burden\n"
            "2. Debt Burden: DTI ratio, revolving utilization rate\n"
            "3. Credit History: account age, derogatory public records, bankruptcies\n"
            "4. Loan Characteristics: grade, sub-grade, purpose, term, amount\n\n"
            "Apply this framework systematically to predict whether a loan will be Fully Paid or Charged Off."
            + _JSON_INSTRUCTION
        ),
        "user_prompt_fn": lambda row: build_user_prompt(row, include_desc=False),
    },
]

# Build few-shot text once (reads raw data — takes a moment)
print("Building few-shot examples from training set...")
few_shot_text = build_few_shot_examples(n_examples=4, random_state=42)
PROMPT_VARIANTS[3]["user_prompt_fn"] = lambda row: (
    "Here are examples of past loan outcomes:\n\n"
    + few_shot_text
    + "\nNow predict the outcome for this new loan application:\n\n"
    + format_loan_features(row, include_desc=False)
)

print(f"\n{len(PROMPT_VARIANTS)} prompt variants defined:")
for v in PROMPT_VARIANTS:
    print(f"  [{v['name']}] {v['description']}")

## Load Data & XGBoost Baseline

In [ ]:
llm_sample = load_llm_sample()
y_true = llm_sample['loan_status'].values

xgb_probs, xgb_preds = run_ml_on_sample(llm_sample)
xgb_metrics = evaluate_predictions(y_true, xgb_preds.tolist(), label="XGBoost (baseline)")

print(f"\nSample: {len(llm_sample)} loans | "
      f"Fully Paid: {(y_true==1).sum()} | Charged Off: {(y_true==0).sum()}")

## Run All Variants

All variants use **GPT-5, no borrower description** — only the prompt changes.  
Experiments run sequentially to avoid rate-limit collisions on a single API key.

In [ ]:
API_PROVIDER = "openai"
MODEL_NAME   = "gpt-5"

all_results = {}
failed = []

for variant in PROMPT_VARIANTS:
    name = variant["name"]
    print(f"\n{'='*60}")
    print(f"Running variant: {name}")
    print(f"{'='*60}")
    try:
        result = run_llm_experiment(
            llm_sample,
            api_provider=API_PROVIDER,
            model_name=MODEL_NAME,
            include_desc=False,
            label=f"GPT-5 | {name}",
            with_logprobs=True,
            system_prompt=variant["system_prompt"],
            user_prompt_fn=variant["user_prompt_fn"],
        )
        all_results[name] = result
    except Exception as e:
        print(f"*** FAILED [{name}]: {e} ***")
        failed.append((name, str(e)))

print(f"\nDone: {len(all_results)} succeeded, {len(failed)} failed")
if failed:
    for name, err in failed:
        print(f"  - {name}: {err}")

## Results Comparison

In [ ]:
rows = [{"variant": "XGBoost (baseline)", "description": "Gradient boosting on 66 structured features", **xgb_metrics}]

for variant in PROMPT_VARIANTS:
    name = variant["name"]
    if name in all_results:
        rows.append({
            "variant": name,
            "description": variant["description"],
            **all_results[name]["metrics"],
        })

summary = pd.DataFrame(rows).set_index("variant")

display_cols = ["accuracy", "auc", "precision_charged_off", "recall_charged_off", "f1_charged_off", "n_valid"]
print(summary[display_cols].to_string(float_format="{:.3f}".format))

In [ ]:
plot_df = summary.reset_index()
plot_df = plot_df[plot_df["variant"] != "XGBoost (baseline)"]  # LLM variants only

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("GPT-5 Prompt Variants vs XGBoost — Same 100-Loan Sample", fontsize=13)

xgb_acc = xgb_metrics["accuracy"]
xgb_f1  = xgb_metrics["f1_charged_off"]
xgb_auc = xgb_metrics.get("auc")

colors = plt.cm.tab10.colors

for ax, (metric, title, xgb_val) in zip(axes, [
    ("accuracy",        "Overall Accuracy",     xgb_acc),
    ("f1_charged_off",  "Charged Off F1",        xgb_f1),
    ("auc",             "AUC (logprobs)",         xgb_auc),
]):
    vals = plot_df[metric].values
    bars = ax.barh(plot_df["variant"], vals, color=colors[:len(plot_df)])
    if xgb_val is not None:
        ax.axvline(xgb_val, color="black", linestyle="--", linewidth=1.2, label=f"XGBoost ({xgb_val:.3f})")
        ax.legend(fontsize=8)
    for bar, val in zip(bars, vals):
        if val is not None and not np.isnan(val):
            ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                    f"{val:.3f}", va="center", fontsize=8)
    ax.set_title(title)
    ax.set_xlim(0, 1.05)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/../../../reports/05_prompt_variance_chart.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# How much does each variant differ from baseline in prediction choices?
baseline_preds = all_results.get("baseline", {}).get("predictions", [])

if baseline_preds:
    print("Prediction delta vs. baseline (how many loans flipped, and net accuracy impact)")
    print("=" * 70)
    for variant in PROMPT_VARIANTS:
        name = variant["name"]
        if name == "baseline" or name not in all_results:
            continue
        preds = all_results[name]["predictions"]
        flipped      = sum(a != b for a, b in zip(baseline_preds, preds))
        improved     = sum((a != t and b == t) for a, b, t in zip(baseline_preds, preds, y_true))
        worsened     = sum((a == t and b != t) for a, b, t in zip(baseline_preds, preds, y_true))
        acc_delta    = all_results[name]["metrics"]["accuracy"] - all_results["baseline"]["metrics"]["accuracy"]
        f1_delta     = all_results[name]["metrics"]["f1_charged_off"] - all_results["baseline"]["metrics"]["f1_charged_off"]
        print(f"\n[{name}]")
        print(f"  Loans flipped vs. baseline: {flipped}/100")
        print(f"  Of those: {improved} improved, {worsened} worsened accuracy")
        print(f"  Accuracy delta: {acc_delta:+.3f} | CO F1 delta: {f1_delta:+.3f}")

## Export Results

Three files saved:
1. **Metrics CSV** — quantitative comparison table
2. **Predictions CSV** — per-loan predictions + reasonings, all variants stacked
3. **Reasonings JSONL** — reasoning text per variant per loan, formatted for promptfoo qualitative eval

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

# 1. Metrics
summary.to_csv(f"{RESULTS_DIR}/05_prompt_variance_metrics.csv")
print(f"Saved: 05_prompt_variance_metrics.csv")

# 2. Predictions — one row per (loan, variant)
pred_rows = []
for variant in PROMPT_VARIANTS:
    name = variant["name"]
    if name not in all_results:
        continue
    result = all_results[name]
    for i, (pred, prob, reasoning) in enumerate(zip(
        result["predictions"], result["probabilities"], result["reasonings"]
    )):
        pred_rows.append({
            "variant":        name,
            "loan_index":     i,
            "actual":         int(y_true[i]),
            "prediction":     pred,
            "correct":        int(pred == y_true[i]) if pred is not None else None,
            "prob_fully_paid": prob,
            "reasoning":      reasoning,
        })

predictions_df = pd.DataFrame(pred_rows)
predictions_df.to_csv(f"{RESULTS_DIR}/05_prompt_variance_predictions.csv", index=False)
print(f"Saved: 05_prompt_variance_predictions.csv ({len(pred_rows)} rows)")

# 3. Reasonings JSONL — one JSON object per loan per variant, for promptfoo
jsonl_path = f"{RESULTS_DIR}/05_prompt_variance_reasonings.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for variant in PROMPT_VARIANTS:
        name = variant["name"]
        if name not in all_results:
            continue
        result = all_results[name]
        for i, (pred, reasoning) in enumerate(zip(result["predictions"], result["reasonings"])):
            record = {
                "variant":    name,
                "loan_index": i,
                "actual":     int(y_true[i]),
                "prediction": pred,
                "correct":    int(pred == y_true[i]) if pred is not None else None,
                "reasoning":  reasoning,
            }
            f.write(json.dumps(record) + "\n")

print(f"Saved: 05_prompt_variance_reasonings.jsonl (for promptfoo qualitative eval)")
print(f"\nAll results in: {RESULTS_DIR}")